# Vector DB Smoke Test

Purpose:
    This notebook runs a compact set of natural-language prompts through the
    reference retriever and displays the returned route, threshold answer,
    internal subqueries, answer scaffold, and retrieved candidates.

Flow:
1. Import `answer_query(...)` from `llm_rag/iii_vector_db/retrieval_engine.py`.
2. Run the smoke prompts against the current reference assets.
3. Summarize route and top-result behavior in a table.
4. Render detailed per-prompt retrieval output for inspection.

This is a notebook inspection workflow, not a formal retrieval benchmark.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    """Locate the repository root from a notebook working directory.

    Args:
        start (Path): Directory where the upward search should begin.

    Returns:
        Path: Repository root containing the `llm_rag/` folder.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "llm_rag").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root containing 'llm_rag'.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EVAL_DIR = REPO_ROOT / "llm_rag" / "evaluation"
VECTOR_DB_DIR = REPO_ROOT / "llm_rag" / "iii_vector_db"
if str(VECTOR_DB_DIR) not in sys.path:
    sys.path.insert(0, str(VECTOR_DB_DIR))

from retrieval_engine import answer_query

print(f"Repo root: {REPO_ROOT}")
print(f"Evaluation folder: {EVAL_DIR}")
print(f"Vector DB folder: {VECTOR_DB_DIR}")


c:\Users\utsav\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer
Evaluation folder: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\llm_rag\evaluation
Vector DB folder: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\llm_rag\iii_vector_db


In [2]:
SMOKE_TEST_PROMPTS = [
    "What supporting information is required for a threatened species assessment?",
    "What additional supporting information is required under specific conditions for threatened taxa?",
    "What are the Criterion B thresholds for EOO and AOO?",
    "How many locations are allowed under Criterion B?",
    "What mature individual thresholds are used under Criterion D?",
    "What extinction probability is used under Criterion E?",
    "What Red List categories count as threatened?",
    "What information should be included in the assessment documentation beyond the basics?",
]

pd.DataFrame({"prompt": SMOKE_TEST_PROMPTS})


,prompt
0,What supporting information is required for a ...
1,What additional supporting information is requ...
2,What are the Criterion B thresholds for EOO an...
3,How many locations are allowed under Criterion B?
4,What mature individual thresholds are used und...
5,What extinction probability is used under Crit...
6,What Red List categories count as threatened?
7,What information should be included in the ass...


In [3]:
def top_result_summary(payload: dict) -> str:
    """Summarize the top retrieved result for the overview table.

    Args:
        payload (dict): Retrieval payload returned by `answer_query(...)`.

    Returns:
        str: Compact source, page, and section summary for the top hit.
    """
    results = payload.get("results") or []
    if not results:
        return "No retrieved results"

    top = results[0]
    meta = top.metadata
    section = meta.get("section_title") or meta.get("table_title") or "Unknown section"
    source_file = meta.get("source_file") or "Unknown source"
    page = meta.get("page")
    return f"{source_file} | p.{page} | {section}"


def run_smoke_tests(prompts: list[str]) -> tuple[list[dict], pd.DataFrame]:
    """Run the retrieval smoke prompts and build an overview table.

    Args:
        prompts (list[str]): Natural-language prompts to send to the retriever.

    Returns:
        tuple[list[dict], pd.DataFrame]: Raw payloads and a tabular summary.
    """
    payloads = []
    rows = []
    for prompt in prompts:
        payload = answer_query(prompt)
        payloads.append(payload)
        rows.append(
            {
                "prompt": prompt,
                "route": payload.get("route"),
                "subquery_count": len(payload.get("subqueries", [])),
                "result_count": len(payload.get("results", [])),
                "threshold_fallback_miss": payload.get("threshold_fallback_miss", False),
                "top_result": top_result_summary(payload),
            }
        )
    return payloads, pd.DataFrame(rows)


payloads, smoke_test_summary = run_smoke_tests(SMOKE_TEST_PROMPTS)
smoke_test_summary


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 490.14it/s, Materializing param=pooler.dense.weight]                               


,prompt,route,subquery_count,result_count,threshold_fallback_miss,top_result
0,What supporting information is required for a ...,hybrid_rag,3,8,False,Required_and_Recommended_Supporting_Informatio...
1,What additional supporting information is requ...,hybrid_rag,3,8,False,Required_and_Recommended_Supporting_Informatio...
2,What are the Criterion B thresholds for EOO an...,deterministic_threshold_lookup,0,0,False,No retrieved results
3,How many locations are allowed under Criterion B?,deterministic_threshold_lookup,0,0,False,No retrieved results
4,What mature individual thresholds are used und...,deterministic_threshold_lookup,0,0,False,No retrieved results
5,What extinction probability is used under Crit...,deterministic_threshold_lookup,0,0,False,No retrieved results
6,What Red List categories count as threatened?,hybrid_rag,1,8,False,RedListGuidelines.pdf | p.11 | Figure 2.1. Str...
7,What information should be included in the ass...,hybrid_rag,3,8,False,Required_and_Recommended_Supporting_Informatio...


In [4]:
def display_payload_details(payload: dict) -> None:
    """Render one retrieval payload in a notebook-friendly format.

    Args:
        payload (dict): Retrieval payload returned by `answer_query(...)`.

    Returns:
        None: The function displays Markdown and tables in the notebook.
    """
    prompt = payload["query"]
    route = payload["route"]
    display(Markdown(f"## Prompt\n\n`{prompt}`"))
    display(Markdown(f"**Route:** `{route}`"))

    if route == "deterministic_threshold_lookup":
        display(Markdown("### Threshold Answer"))
        display(Markdown(payload["threshold_answer"].replace("\n", "  \n")))
        return

    if payload.get("threshold_fallback_miss"):
        display(Markdown("_Threshold lookup was attempted but did not match exactly, so the query fell back to hybrid retrieval._"))

    display(Markdown("### Internal Retrieval Queries"))
    for subquery in payload.get("subqueries", []):
        display(Markdown(f"- {subquery}"))

    display(Markdown("### Answer Scaffold"))
    display(Markdown(payload.get("answer_scaffold", "No scaffold returned").replace("\n", "  \n")))

    results = payload.get("results", [])
    if not results:
        display(Markdown("### Results\n\nNo retrieved candidates returned."))
        return

    detail_rows = []
    for rank, candidate in enumerate(results, start=1):
        meta = candidate.metadata
        detail_rows.append(
            {
                "rank": rank,
                "source_file": meta.get("source_file"),
                "page": meta.get("page"),
                "block_type": meta.get("block_type"),
                "section_title": meta.get("section_title"),
                "table_title": meta.get("table_title"),
                "table_id": meta.get("table_id"),
                "row_id": meta.get("row_id"),
                "fallback": meta.get("fallback"),
                "forced_for_coverage": candidate.forced_for_coverage,
                "dense_score": round(candidate.dense_score, 4),
                "bm25_score": round(candidate.bm25_score, 4),
                "rerank_score": round(candidate.rerank_score, 4),
                "snippet": candidate.text[:220].replace("\n", " "),
            }
        )

    display(Markdown("### Retrieved Candidates"))
    display(pd.DataFrame(detail_rows))


for payload in payloads:
    display_payload_details(payload)


## Prompt

`What supporting information is required for a threatened species assessment?`

**Route:** `hybrid_rag`

### Internal Retrieval Queries

- required supporting information under all conditions for IUCN Red List assessments

- required supporting information under specific conditions for threatened taxa

...

- What information should be included in the assessment documentation beyond the basics?

### Answer Scaffold

Answer scaffold  
---------------  
Additional required under specific conditions relevant to threatened taxa:  
- Entry 10: 10. Time period over For taxa listed as • To justify the Red Record this in SIS as the which 3-generation threatened under List Category and start year for the 3- decline is measured criterion A4 Criteria used generation time period. around the prese  
- Entry 9: 9. Generation length For taxa listed as • To justify the Red For definition of generation threatened under List Category and length refer to the current criteria A and C1 Criteria used version of the Guidelines for Using the IUCN Red List Categories a  
- Entry 12: 12. Coding and For taxa listed as Near • To justify the Red justification of the Threatened List Category and criteria that are nearly Criteria used met or the reasons for the classification (e.g., dependence on ongoing conservation measures) 8  
- Entry 15: 15. Narrative text about For taxa listed as • To justify the Red Required for supporting the the geographic range, Extinct, Extinct in the List Category and assessment with contextual population, habitat and Wild, Critically Criteria used and explana  
  
Required for all assessments:  
- Page 3 | Entry 1: 1. Scientific name1 • To identify which taxon is If the taxon is already in SIS, this being assessed information requires no additional effort from the Assessors. If the taxon is not • To support Red List website yet recorded in SIS, Assessor  
- Page 3 | Entry 2: 2. Higher taxonomy details • To identify which taxon is If the taxon is already in SIS, this (Kingdom, Phylum, Class, being assessed requires no additional effort from the Order, Family) Assessors. If the taxon is not yet • To support Red Lis  
- Page 3 | Entry 3: 3. Taxonomic authorities for all • To identify which taxon is If the taxon is already in SIS, this specific and infra-specific being assessed information requires no additional effort names used, following the from the Assessors. If the taxon  
- Page 3 | Entry 4: 4. IUCN Red List Category and • To identify the current status The Red List Category and Criteria Criteria (including sub- of the taxon represent the most fundamental criteria) met at the highest elements of a Red List assessment. • To suppor

### Retrieved Candidates

,rank,source_file,page,block_type,section_title,table_title,table_id,row_id,fallback,forced_for_coverage,dense_score,bm25_score,rerank_score,snippet
0,1,Required_and_Recommended_Supporting_Informatio...,3,table,Required Information Purpose Guidance Notes,Table 1: Required supporting information for a...,table_1,NaN,True,True,0.0000,0.0000,0.0000,Table 1: Required supporting information for a...
1,2,Required_and_Recommended_Supporting_Informatio...,8,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,10.0,True,False,0.5930,24.7786,1.8487,Entry 10: 10. Time period over For taxa listed...
2,3,Required_and_Recommended_Supporting_Informatio...,8,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,9.0,True,False,0.5829,24.7719,1.8420,Entry 9: 9. Generation length For taxa listed ...
3,4,Required_and_Recommended_Supporting_Informatio...,8,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,12.0,True,False,0.5880,24.2433,1.8411,Entry 12: 12. Coding and For taxa listed as Ne...
4,5,Required_and_Recommended_Supporting_Informatio...,10,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,15.0,True,False,0.6398,21.1173,1.8316,Entry 15: 15. Narrative text about For taxa li...
5,6,Required_and_Recommended_Supporting_Informatio...,8,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,11.0,True,False,0.5817,23.4331,1.8182,"Entry 11: 11. The data, For taxa listed under ..."
6,7,Required_and_Recommended_Supporting_Informatio...,9,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,14.0,True,False,0.6114,18.5058,1.8112,Entry 14: 14. Major threats to the For taxa li...
7,8,Required_and_Recommended_Supporting_Informatio...,9,table_row,Specific Condition Purpose Guidance Notes,Table 2: Required supporting information under...,table_2,13.0,True,False,0.5892,20.1175,1.7969,Entry 13: 13. Taxonomic notes • For taxa previ...
